<a href="https://colab.research.google.com/github/JohnMaleek/ML/blob/main/Assignment_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Exercise 1
Use the Keras "Sequential" functionality to build `model_2` with the following specifications:

* Two hidden layers.
* First hidden layer of size 400 and second of size 300.
* Dropout of .4 at each layer.
* How many parameters does your model have?  How does it compare with the previous model?
* Train this model for 20 epochs with RMSProp at a learning rate of .001 and a batch size of 128.
(2p)

In [1]:


# Import required libraries
import tensorflow as tf
import numpy as np
import time
from keras.models import Sequential
from keras.layers import Dense, Dropout
from keras.optimizers import RMSprop, SGD
from keras.utils import to_categorical

# Load and prepare CIFAR-10 dataset
(X_train, Y_train), (X_test, Y_test) = tf.keras.datasets.cifar10.load_data()

# Reshape images to feature vectors
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1] * X_train.shape[2] * X_train.shape[3])
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1] * X_test.shape[2] * X_test.shape[3])
Y_train = Y_train.ravel()
Y_test = Y_test.ravel()

# Scale pixel values to [0, 1]
X_train = np.array(X_train) / 255.0
X_test = np.array(X_test) / 255.0

# Convert labels to binary vectors
classes = 10
Y_train_vectors = to_categorical(Y_train, classes)
Y_test_vectors = to_categorical(Y_test, classes)

print("[INFO] Dataset loaded and preprocessed")
print("[INFO] X_train shape: {}".format(X_train.shape))
print("[INFO] X_test shape: {}".format(X_test.shape))

# Build ORIGINAL model for comparison
#
model_original = Sequential()
model_original.add(Dense(500, input_dim=3072, activation="relu"))
model_original.add(Dense(10, activation='softmax'))

print("\n[INFO] ORIGINAL MODEL (3072-500-10):")
model_original.summary()

#
# Build model_2 with two hidden layers and dropout (3072-400-300-10)
#
model_2 = Sequential()
model_2.add(Dense(400, input_dim=3072, activation="relu"))
model_2.add(Dropout(0.4))
model_2.add(Dense(300, activation="relu"))
model_2.add(Dropout(0.4))
model_2.add(Dense(10, activation='softmax'))

print("\n[INFO] MODEL_2 (3072-400-300-10 with Dropout):")
model_2.summary()

# Parameter comparison
original_params = model_original.count_params()
model_2_params = model_2.count_params()
print("\n[INFO] PARAMETER COMPARISON:")
print("    Original model params:   {:,}".format(original_params))
print("    Model_2 params:          {:,}".format(model_2_params))
print("    Difference:              {:,}".format(model_2_params - original_params))

# Compile with RMSProp
print("\n[INFO] compiling model_2...")
rmsprop = RMSprop(learning_rate=0.001)
model_2.compile(loss="mse", optimizer=rmsprop, metrics=["accuracy"])

# Train for 20 epochs with batch_size=128
print("[INFO] training model_2...")
start = time.time()
model_2.fit(X_train, Y_train_vectors, epochs=20, batch_size=128, verbose=1)
print("[INFO] training time: {:.2f}s".format(time.time() - start))

# Evaluate on test set
print("[INFO] evaluating model_2 on test set...")
(loss, accuracy) = model_2.evaluate(X_test, Y_test_vectors, batch_size=128, verbose=1)
print("[INFO] loss={:.4f}, accuracy: {:.2f}%".format(loss, accuracy * 100))

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step
[INFO] Dataset loaded and preprocessed
[INFO] X_train shape: (50000, 3072)
[INFO] X_test shape: (10000, 3072)

[INFO] ORIGINAL MODEL (3072-500-10):


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 500)            │     1,536,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 10)             │         5,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,541,510 (5.88 MB)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_2 (Dense)                 │ (None, 400)            │     1,229,200 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 400)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 300)            │       120,300 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 300)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 10)             │         3,010 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,352,510 (5.16 MB)

 Trainable params: 1,352,510 (5.16 MB)

 Non-trainable params: 0 (0.00 B)


[INFO] PARAMETER COMPARISON:
    Original model params:   1,541,510
    Model_2 params:          1,352,510
    Difference:              -189,000

[INFO] compiling model_2...
[INFO] training model_2...
Epoch 1/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 25ms/step - accuracy: 0.2329 - loss: 0.0852
Epoch 2/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.3089 - loss: 0.0809
Epoch 3/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 30ms/step - accuracy: 0.3334 - loss: 0.0788
Epoch 4/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.3574 - loss: 0.0772
Epoch 5/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 20s 28ms/step - accuracy: 0.3690 - loss: 0.0761
Epoch 6/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 10s 25ms/step - accuracy: 0.3811 - loss: 0.0752
Epoch 7/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.3890 - loss: 0.0745
Epoch 8/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 12s 31ms/step - accuracy: 0.4003 - loss: 0.0736
Epoch 9/20
391/391 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - accuracy: 0.4068 - loss: 0.0732
Epoch 

### Exercise 2
Suppose that we have an ANN with 3 output neurons and their weighted inputs are $z_1 = 0.4$, $z_2=0.9$, and $z_3=-0.5$. What are the output values if we use sigmoid and softmax activation functions? (1p)

In [2]:
#
# Exercise 2: Sigmoid and Softmax activation functions
#

# Import required libraries
import numpy as np

# Given weighted inputs
z = np.array([0.4, 0.9, -0.5])

# Sigmoid activation:
sigmoid_output = 1 / (1 + np.exp(-z))
print("[INFO] weighted inputs: z = {}".format(z))
print("[INFO] Sigmoid outputs:")
for i, out in enumerate(sigmoid_output):
    print("    σ(z{}) = {:.4f}".format(i+1, out))

# Softmax activation:
exp_z = np.exp(z)
softmax_output = exp_z / np.sum(exp_z)
print("[INFO] Softmax outputs:")
for i, out in enumerate(softmax_output):
    print("    a{} = {:.4f}".format(i+1, out))

# Verify softmax sums to 1
print("[INFO] Sum of softmax outputs: {:.4f}".format(np.sum(softmax_output)))

[INFO] weighted inputs: z = [ 0.4  0.9 -0.5]
[INFO] Sigmoid outputs:
    σ(z1) = 0.5987
    σ(z2) = 0.7109
    σ(z3) = 0.3775
[INFO] Softmax outputs:
    a1 = 0.3273
    a2 = 0.5396
    a3 = 0.1331
[INFO] Sum of softmax outputs: 1.0000
